# EfficientViT B2 + forensic fusion: letterbox 1024×1024

Запускает `configs/efficientvit_b2_letterbox.yaml`: EfficientViT MIT B2 (ImageNet r288), вход 1024×1024 с forensic-картами и настройками split/обучения из конфига.

Для нового эксперимента скопируйте YAML, измените параметры и путь ниже. Результаты: `runs/<run_name>/summary.json`, `metrics.csv`, `notes.md`; лучшие веса: `ckpt/best.pt`.


Длинная сторона масштабируется до 1024, пропорции сохраняются с округлением до пикселя. Padding справа/снизу. Поля исключены из segmentation loss и classification pooling; перед валидацией и submission они обрезаются.

DCT извлекается до resize; карты размещаются на той же геометрии. Вход сети остаётся квадратным, поэтому padding не уменьшает FLOPs.


Полная модель: **97.008 GFLOPS**, 16.31 млн параметров. Форензика добавляется к encoder features на stride 8/16/32; U-Net decoder и обе головы сохранены.

Batch 2 × accumulation 8 = 16, бюджет 192 000 примеров, тот же fold и mixed/original протокол.

Pretrained-веса `timm/efficientvit_b2.r288_in1k` загружаются при первом запуске. Скорость на H100 и пиковая память обучения на RTX 3070 ещё не измерены. При нехватке памяти используйте batch 1 × accumulation 16 в копии YAML с новым run_name.


In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config import load_experiment_config
from src.training.engine import run_experiment

cfg = load_experiment_config(project_root / "configs" / "efficientvit_b2_letterbox.yaml")
cfg

In [ ]:
run = run_experiment(cfg)
run.summary